# 02. 첨부 CSV로 실행하는 유찰 위험도 분석

Python 3에서 `pandas`, `numpy`, `scipy`를 설치하세요. 필요 시 `%pip install pandas numpy scipy`를 별도 셀에서 실행합니다.

**입력:** 정제 공고 CSV와 정제 입찰결과 CSV 두 개만 필요합니다. 첫 코드 셀의 `DATA_DIR`에 폴더를 지정한 뒤 모든 셀을 위에서부터 실행하세요. 기존 01번 정제 노트북은 필요하지 않습니다.

공고의 `noticeKey`로 결과를 연결하고 공고일 기준 2021~2024년 학습, 2025년 검증 자료를 직접 생성합니다. 결과 중복은 공고 단위로 집계하고 상충 상태·취소공고·수의계약·미연결 결과는 제외합니다. 결과 파일은 `risk_review/`에 저장됩니다.

기관그룹 매핑이 없어 기관코드를 대체 기준으로 유지합니다. 계약정보·조달계획의 사후 상태는 위험도에 사용하지 않습니다.


## 1. 학습·검증 자료와 숫자형 변수 구간

아래 코드가 첨부 CSV를 연결하여 취소공고, 수의계약, 결과 미확인, 판단보류 등을 제외하고 분석용 자료를 생성합니다. `train`(2021~2024년)은 **변수 선정과 과거 유찰률표 작성**에만, `valid`(2025년)은 **미래 공고를 흉내 낸 평가**에만 사용합니다. `is_failed`는 유찰 `1`, 낙찰·순위확정 `0`인 결과값입니다.

카이제곱 검정은 범주별 건수를 비교하므로 연속값을 미리 정한 구간으로 바꿉니다. 2025년 결과를 보고 경계를 고르지 않았습니다.

| 원래 값 | 새 범주 | 구간 |
|---|---|---|
| 공고일~참가등록마감(`notice_duration_days`) | `duration_bin` | 5일 이하 / 5일 초과~10일 / 10일 초과 / 오류·결측 |
| 기초금액(`bsicExpt`) | `amount_bin` | 1천만원 이하 / 1천만원 초과~1억원 / 1억원 초과 / 오류·결측 |
| 공고 분기(`quarter`) | `quarter_bin` | Q1 / Q2 / Q3 / Q4 |

공고기간에는 시각이 포함되어 5.5일 같은 값이 가능합니다. `5일 초과~10일`은 5일보다 크고 10일 이하인 값입니다. 음수 기간은 오류로 표시합니다. **아래 출력에서** 학습·검증 건수 및 오류·결측 건수를 확인하세요.

In [3]:
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

START = perf_counter()
# Windows에서는 필요 시 다음처럼 지정하세요.
DATA_DIR = Path(r"C:\workspace\defense-procurement-risk-dashboard\data\clean")
# None이면 현재 폴더, clean/, upload/, ../upload/에서 두 CSV를 찾습니다.
OUT = Path("risk_review")
OUT.mkdir(parents=True, exist_ok=True)
NOTICE_FILE = "bid_notice_clean_20210101_20251231.csv"
RESULT_FILE = "bid_result_clean_20210101_20251231.csv"
candidates = [Path(DATA_DIR)] if DATA_DIR is not None else [Path.cwd(), Path("clean"), Path("upload"), Path("../upload")]
DATA = next((p for p in candidates if (p/NOTICE_FILE).is_file() and (p/RESULT_FILE).is_file()), None)
if DATA is None:
    raise FileNotFoundError("공고·결과 정제 CSV 두 개를 같은 폴더에 넣고 DATA_DIR에 해당 경로를 지정하세요.")

def read_clean(path, required):
    d = pd.read_csv(path, dtype="string", encoding="utf-8-sig", low_memory=False)
    missing = set(required) - set(d.columns)
    if missing:
        raise ValueError(f"{path.name}: 필수 컬럼 누락 {sorted(missing)}")
    for c in d.columns:
        d[c] = d[c].str.strip().replace("", pd.NA)
    return d

notice = read_clean(DATA/NOTICE_FILE, ["noticeKey", "pblancDate", "bidPartcptRegistClosDt", "orntCode", "busiDivs", "cntrctMth", "bsicExpt", "pblancSe"])
result = read_clean(DATA/RESULT_FILE, ["noticeKey", "bidResult", "opengDt"])
if notice.noticeKey.isna().any() or notice.noticeKey.duplicated().any():
    raise ValueError("공고 noticeKey가 결측 또는 중복입니다. 공고 정제 결과를 먼저 확인하세요.")
# 날짜는 문자열 길이에 맞는 포맷으로 명시적으로 변환합니다.
notice["pblancDate"] = pd.to_datetime(notice.pblancDate, format="%Y%m%d", errors="coerce")
notice["registration_deadline"] = pd.to_datetime(notice.bidPartcptRegistClosDt, format="%Y%m%d%H%M", errors="coerce")
result["result_open_dt"] = pd.to_datetime(result.opengDt, format="%Y%m%d%H%M", errors="coerce")
result["label"] = result.bidResult.map({"유찰": 1, "낙찰": 0, "순위확정": 0}).astype("Int64")
# 공고에 여러 품목 결과가 있어도 공고 한 건으로 집계합니다.
# 상태 원문이 서로 다르거나 미정의 상태가 하나라도 있으면 판단보류합니다.
result_summary = result.groupby("noticeKey", dropna=False).agg(
    result_rows=("bidResult", "size"),
    result_state_count=("bidResult", "nunique"),
    label_count=("label", "count"),
    is_failed=("label", "first"),
    result_open_dt=("result_open_dt", "max"),
).reset_index()
result_summary["ambiguous_result"] = result_summary.result_state_count.ne(1) | result_summary.label_count.ne(result_summary.result_rows)
result_summary.loc[result_summary.ambiguous_result, "is_failed"] = pd.NA
merged = notice.rename(columns={"noticeKey": "notice_key"}).merge(
    result_summary.rename(columns={"noticeKey": "notice_key"}),
    on="notice_key", how="left", validate="one_to_one", indicator=True)
merged["year"] = merged.pblancDate.dt.year.astype("Int64")
merged["quarter"] = merged.pblancDate.dt.quarter.astype("Int64")
merged["notice_duration_days"] = (merged.registration_deadline - merged.pblancDate).dt.total_seconds()/86400
merged["bsicExpt"] = pd.to_numeric(merged.bsicExpt.str.replace(",", "", regex=False), errors="coerce")

# 서로 겹치지 않는 제외 사유를 부여하여 입력→제외→분석 건수가 맞는지 확인합니다.
reason = pd.Series("포함", index=merged.index, dtype="string")
def exclude(mask, label):
    reason.loc[reason.eq("포함") & mask.fillna(False)] = label
exclude(merged.pblancDate.isna(), "공고일 오류/결측")
exclude(~merged.year.between(2021, 2025), "공고연도 범위 밖")
exclude(merged.pblancSe.eq("취소공고"), "취소공고")
exclude(merged.cntrctMth.eq("수의계약"), "수의계약")
exclude(merged._merge.eq("left_only"), "결과 미연결")
exclude(merged.is_failed.isna(), "결과 판단보류")
exclude(merged.result_open_dt.isna(), "개찰일 오류/결측")
exclude(merged.result_open_dt.lt(merged.pblancDate), "개찰일이 공고일보다 빠름")
# 2025년 평가 시점 이후 개찰된 결과를 과거 유찰률표에 넣지 않습니다.
# 결과 공개일은 제공되지 않았으므로 개찰일을 이용한 시점 통제라는 한계가 있습니다.
exclude(merged.year.between(2021, 2024) & merged.result_open_dt.ge(pd.Timestamp("2025-01-01")), "학습 기준일 이후 개찰")
exclude(merged.year.eq(2025) & merged.result_open_dt.ge(pd.Timestamp("2026-01-01")), "검증 관측기간 이후 개찰")
merged["exclusion_reason"] = reason
quality = reason.value_counts().rename_axis("reason").reset_index(name="notice_count")
quality.to_csv(OUT/"analysis_exclusion_summary.csv", index=False, encoding="utf-8-sig")
merged.loc[reason.ne("포함"), ["notice_key", "pblancDate", "exclusion_reason"]].to_csv(OUT/"excluded_notices.csv", index=False, encoding="utf-8-sig")
analysis = merged.loc[reason.eq("포함")].drop(columns="_merge").copy()
analysis["is_failed"] = analysis.is_failed.astype(int)
train = analysis.loc[analysis.year.between(2021, 2024)].copy().reset_index(drop=True)
valid = analysis.loc[analysis.year.eq(2025)].copy().reset_index(drop=True)
if train.empty or valid.empty:
    raise ValueError("학습 또는 검증 데이터가 비어 있습니다. 제외 사유를 확인하세요.")
train.to_csv(OUT/"analysis_train_2021_2024.csv", index=False, encoding="utf-8-sig")
valid.to_csv(OUT/"analysis_valid_2025.csv", index=False, encoding="utf-8-sig")
assert train.notice_key.is_unique and valid.notice_key.is_unique
assert not set(train.notice_key) & set(valid.notice_key)
assert train.is_failed.isin([0,1]).all() and valid.is_failed.isin([0,1]).all()

def bin_features(df):
    """검증 결과를 보지 않고 고정한 구간으로 변환합니다."""
    d = df.copy()
    days = pd.to_numeric(d["notice_duration_days"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    amount = pd.to_numeric(d["bsicExpt"], errors="coerce").to_numpy(dtype=float, na_value=np.nan)
    d["duration_bin"] = np.select(
        [(days >= 0) & (days <= 5), (days > 5) & (days <= 10), days > 10],
        ["5일 이하", "5일 초과~10일", "10일 초과"], default="기간 오류/결측")
    d["amount_bin"] = np.select(
        [(amount >= 0) & (amount <= 10_000_000), (amount > 10_000_000) & (amount <= 100_000_000), amount > 100_000_000],
        ["1천만원 이하", "1천만원 초과~1억원", "1억원 초과"], default="금액 오류/결측")
    d["quarter_bin"] = "Q" + d.quarter.astype("Int64").astype("string")
    return d
train, valid = bin_features(train), bin_features(valid)
print("입력 폴더:", DATA.resolve())
print(quality.to_string(index=False))
print(f"학습: {len(train):,}건 / 검증: {len(valid):,}건")
print("학습 공고기간 오류/결측:", train.duration_bin.eq("기간 오류/결측").sum())
print("학습 기초금액 오류/결측:", train.amount_bin.eq("금액 오류/결측").sum())
print(f"자료 준비 시간: {perf_counter()-START:.2f}초")


입력 폴더: C:\workspace\defense-procurement-risk-dashboard\data\clean
      reason  notice_count
          포함         34071
      결과 미연결         15505
        수의계약          5867
        취소공고          5552
   공고연도 범위 밖           119
학습 기준일 이후 개찰            22
     결과 판단보류             6
학습: 27,852건 / 검증: 6,219건
학습 공고기간 오류/결측: 4
학습 기초금액 오류/결측: 573
자료 준비 시간: 1.15초


## 2. 카이제곱 독립성 검정과 변수 선정

학습 자료에서 각 변수의 범주별 **유찰(`1`)과 비유찰(`0`) 건수**를 비교합니다. p값이 작으면 관계가 있다는 증거지만, **유찰의 원인이라는 뜻은 아닙니다.** 표본이 클 때 작은 차이도 유의해질 수 있으므로 연관성의 크기인 `Cramér's V`도 함께 기록합니다.

- 기존 기준: 기관코드, 사업유형, 계약·경쟁방식.
- 추가 후보: 공고기간 구간, 기초금액 구간, 공고 분기.
- 추가 후보 세 개의 p값에 **Holm 보정**을 적용합니다. 보정 p값이 0.05 미만이면 비교할 위험도 조합에 포함합니다. 기존 세 변수는 이 보정의 대상이 아닙니다.
- 기간 오류·결측과 금액 오류·결측은 각 숫자 구간의 검정에서 제외하지만 원본 공고를 삭제하지 않습니다.
- `expected_lt5`는 기대빈도가 5 미만인 셀 수입니다. 이 값이 크면 근사 카이제곱 p값을 조심해서 해석해야 합니다. 기관코드처럼 범주가 많을 때 특히 그렇습니다.

**아래 출력의 `selected_additional`**이 추가 후보의 채택 여부입니다. 보정 p값이 작더라도 운영 위험도에 최종 채택할지는 2025년 결과와 표본 부족을 함께 확인합니다.

In [4]:
BASE = ['orntCode', 'busiDivs', 'cntrctMth']
ADDITIONAL = ['duration_bin', 'amount_bin', 'quarter_bin']
LABELS = {
    'orntCode':'기관코드 (기관그룹 대체)', 'busiDivs':'사업유형',
    'cntrctMth':'계약/경쟁방식', 'duration_bin':'공고기간 구간',
    'amount_bin':'기초금액 구간', 'quarter_bin':'공고 분기',
}

def test_feature(df, col):
    """변수별 범주와 유찰(0/1)의 독립성 및 연관성 크기를 보고한다."""
    subset = df
    # 오류/결측은 수치 구간 자체가 아니므로 검정에서는 제외, 예측 때는 상위 조합으로 대체.
    if col == 'duration_bin':
        subset = subset[subset[col] != '기간 오류/결측']
    if col == 'amount_bin':
        subset = subset[subset[col] != '금액 오류/결측']
    subset = subset[subset[col].notna()]
    table = pd.crosstab(subset[col], subset['is_failed'])
    if table.shape[0] < 2 or table.shape[1] < 2:
        return dict(variable=LABELS[col], column=col, n=len(subset), chi2=np.nan,
                    dof=np.nan, p_value=np.nan, cramers_v=np.nan,
                    expected_lt5=np.nan, min_expected=np.nan)
    chi2, p, dof, exp = chi2_contingency(table, correction=False)
    # 기대빈도가 너무 작은 셀이 많으면 근사 카이제곱 p값의 해석에 주의해야 합니다.
    return dict(variable=LABELS[col], column=col, n=len(subset), chi2=chi2,
                dof=dof, p_value=p,
                cramers_v=np.sqrt(chi2/(len(subset)*min(table.shape[0]-1, table.shape[1]-1))),
                expected_lt5=int((exp < 5).sum()), min_expected=exp.min())

tests = pd.DataFrame([test_feature(train, c) for c in BASE+ADDITIONAL])
# 추가 후보 3개의 다중 검정에 Holm 보정 적용. 기준변수는 별도 표기.
p = tests.loc[tests.column.isin(ADDITIONAL), 'p_value'].dropna().sort_values()
adjusted = pd.Series(index=p.index, dtype='float64')
running = 0.0
for rank, (idx, value) in enumerate(p.items()):
    # 가장 작은 p부터 3, 2, 1을 곱한 후 앞선 보정값보다 작지 않게 만듭니다.
    running = max(running, min(1.0, value * (len(ADDITIONAL)-rank)))
    adjusted.loc[idx] = running
tests['holm_p_additional'] = adjusted
tests['selected_additional'] = tests.column.isin(ADDITIONAL) & tests.holm_p_additional.lt(.05)
selected = [c for c in ADDITIONAL if bool(tests.set_index('column').loc[c, 'selected_additional'])]
print(tests.to_string(index=False))
print('추가 선정:', selected)
print('기존 기준변수 중 p>=0.05 또는 검정 불가:', tests.loc[tests.column.isin(BASE) & (tests.p_value.ge(.05) | tests.p_value.isna()), 'column'].tolist())


      variable       column     n        chi2  dof       p_value  cramers_v  expected_lt5  min_expected  holm_p_additional  selected_additional
기관코드 (기관그룹 대체)     orntCode 27852 1506.256046  208 2.023693e-195   0.232553           124      0.177079                NaN                False
          사업유형     busiDivs 27852   50.854402    1  9.947794e-13   0.042730             0   1050.962947                NaN                False
       계약/경쟁방식    cntrctMth 27852   48.845525    6  8.006007e-09   0.041878             1      1.947867                NaN                False
       공고기간 구간 duration_bin 27848   12.270579    2  2.165098e-03   0.020991             0    117.065929       2.165098e-03                 True
       기초금액 구간   amount_bin 27279   88.712250    2  5.449790e-20   0.057027             0    847.246087       1.634937e-19                 True
         공고 분기  quarter_bin 27852   16.441197    3  9.206376e-04   0.024296             0    886.987936       1.841275e-03              

## 3. 과거 유찰률로 위험도 계산하기

위험도 = **동일 조건 과거 유찰 공고 수 ÷ 동일 조건 과거 전체 공고 수**. 예를 들어 과거 20건 중 6건이 유찰이면 위험도는 `0.30`(30%)입니다. CSV에는 `30`이 아닌 **0.30 비율**로 저장됩니다.

가장 세분화된 조합의 과거 사례가 **10건 미만**이면 상위 조합으로 올라갑니다. 공고기간·기초금액이 선택된 경우의 순서는 다음과 같습니다.

1. 기관코드 × 사업유형 × 계약방식 × 공고기간 × 기초금액
2. 기관코드 × 사업유형 × 계약방식 × 공고기간
3. 기관코드 × 사업유형 × 계약방식
4. 기관코드 × 사업유형
5. 기관코드
6. 전체 과거 공고

분기가 추가되면 분기를 가장 먼저 제거합니다. 기간·금액의 입력이 오류 또는 결측이면 해당 구간으로 위험도를 계산하지 않고 상위 조합을 사용합니다.

`risk_basis_level`: 실제로 적용한 단계. `risk_basis_count`: 그 단계의 과거 근거 건수. `full_level_count`: 처음 시도한 최상위 조합의 과거 건수. `fallback_reason`: **10건 미만**인지 **입력 오류·결측**인지 구분하는 사유입니다.

코드의 `history`만 유찰률 계산에 사용합니다. `future`의 실제 결과값은 위험도에 섞이지 않으며 뒤의 평가에만 사용합니다.

In [5]:
MIN_N = 10
def score_by_history(history, future, extra):
    """신규 공고의 결과값을 쓰지 않고 과거 공고만 집계한다."""
    # 예: 추가 변수 [기간, 금액]이면 5변수 → 4변수 → 기본 3변수 → 2변수
    # → 기관 1변수 → 전체 평균 순으로 10건 이상인 첫 집계를 선택합니다.
    history = history.reset_index(drop=True)
    future = future.reset_index(drop=True)
    if history.empty:
        raise ValueError('과거 데이터가 없습니다.')
    levels = [BASE + extra[:i] for i in range(len(extra), -1, -1)]
    levels += [BASE[:2], BASE[:1]]
    output = future[['notice_key']].copy().reset_index(drop=True)
    if 'is_failed' in future:
        output['is_failed'] = future['is_failed'].to_numpy()
    output['risk_score'] = np.nan
    output['risk_basis_count'] = pd.Series(pd.NA, index=output.index, dtype='Int64')
    output['risk_basis_level'] = pd.Series(pd.NA, index=output.index, dtype='string')
    output['basis_step'] = pd.Series(pd.NA, index=output.index, dtype='Int64')
    output['full_level_count'] = pd.Series(pd.NA, index=output.index, dtype='Int64')
    output['fallback_reason'] = pd.Series(pd.NA, index=output.index, dtype='string')
    # pending=True는 아직 적용할 수 있는 10건 이상 과거 조합을 찾지 못한 공고입니다.
    pending = np.ones(len(output), dtype=bool)
    for step, cols in enumerate(levels):
        # history만 집계하므로 2025년 결과가 위험도 계산에 섞이지 않습니다.
        table = history.groupby(cols, dropna=False)['is_failed'].agg(['sum', 'count']).reset_index()
        # NaN으로 그룹을 만들어도 NaN 키는 미래 공고에서 의도적으로 상위 조합으로 이동.
        match = future.loc[pending, cols].merge(table, on=cols, how='left', sort=False)
        # pandas 설정에 따라 반환 배열이 읽기 전용일 수 있습니다.
        # bool 배열을 복사하고, 조건 추가 시 새 배열을 만들어 대입합니다.
        ok = match['count'].ge(MIN_N).fillna(False).to_numpy(dtype=bool, copy=True)
        if step == 0:
            # 최초 조합의 표본 수를 보존해야 '10건 미만'과 오류/결측을 구분할 수 있습니다.
            output['full_level_count'] = pd.array(match['count'].fillna(0), dtype='Int64')
            invalid = np.zeros(len(output), dtype=bool)
            if 'duration_bin' in cols:
                invalid |= match['duration_bin'].eq('기간 오류/결측').fillna(False).to_numpy()
            if 'amount_bin' in cols:
                invalid |= match['amount_bin'].eq('금액 오류/결측').fillna(False).to_numpy()
            output.loc[invalid, 'fallback_reason'] = '입력 오류/결측'
            output.loc[~invalid & ~ok, 'fallback_reason'] = '최상위 조합 10건 미만'
        # 어느 단계에서도 결측 기준변수와 오류 구간은 적용하지 않습니다.
        ok = ok & match[cols].notna().all(axis=1).to_numpy()
        if step == 0:
            invalid |= ~match[cols].notna().all(axis=1).to_numpy()
            output.loc[invalid, 'fallback_reason'] = '입력 오류/결측'
        if extra and step < len(extra):
            # 공고기간·가격 원문 오류/결측의 비슷한 오류끼리로 위험도를 계산하지 않는다.
            for c in cols:
                if c == 'duration_bin': ok = ok & match[c].ne('기간 오류/결측').fillna(False).to_numpy()
                if c == 'amount_bin': ok = ok & match[c].ne('금액 오류/결측').fillna(False).to_numpy()
        idx = np.flatnonzero(pending)[ok]
        # 위험도는 같은 조합의 과거 유찰 수 / 과거 공고 수 (0~1 비율)입니다.
        output.loc[idx, 'risk_score'] = (match.loc[ok, 'sum']/match.loc[ok, 'count']).to_numpy()
        output.loc[idx, 'risk_basis_count'] = match.loc[ok, 'count'].to_numpy()
        output.loc[idx, 'risk_basis_level'] = ' × '.join(LABELS[c] for c in cols)
        output.loc[idx, 'basis_step'] = step
        pending[idx] = False
    if pending.any():
        idx = np.flatnonzero(pending)
        output.loc[idx, 'risk_score'] = history.is_failed.mean()
        output.loc[idx, 'risk_basis_count'] = len(history)
        output.loc[idx, 'risk_basis_level'] = '전체'
        output.loc[idx, 'basis_step'] = len(levels)
    assert output.risk_score.between(0, 1).all()
    assert output.notice_key.is_unique
    assert output['risk_score'].notna().all()
    return output

## 4. 2025년 검증 지표와 10건 미만 대체율

`기준`은 원래 세 변수, `추가_1`부터는 검정에서 선정된 추가 변수를 기간→금액→분기 순으로 하나씩 더한 조합입니다. 선정 결과에 따라 조합 수와 내용이 바뀝니다.

| 출력 컬럼 | 읽는 방법 |
|---|---|
| `sparse_fallback_rate` | 최상위 조합의 과거 사례가 **10건 미만**이라 대체한 공고 비율. 이번 지적의 핵심 지표 |
| `invalid_fallback_count` | 기간·금액의 오류 또는 결측으로 대체한 공고 수. 위의 10건 미만 비율과 분리 |
| `fallback_rate` | 두 사유를 합친 전체 대체 비율 |
| `agency_only_or_global_rate` | 실제로 기관코드 단독 또는 전체 평균까지 내려간 공고 비율 |

**성과 지표:** 2025년의 실제 검증 공고를 위험도 순으로 정렬하고 상위 20%를 선택합니다(검증 건수×0.2를 내림). `top20_capture`는 이 고위험군에 포함된 실제 유찰 건수 ÷ 2025년 전체 실제 유찰 건수입니다. `top20_failure_rate`는 고위험군 자체의 유찰률입니다. 위험도가 동점이면 `notice_key` 순서로 고정합니다.

전체 대체 비율이 높아져도 전부 기관 평균으로 간 것은 아닐 수 있습니다. 그래서 **실제 기관 단독/전체 적용 비율**을 별도로 보여 줍니다. 변수의 통계적 유의성과 2025년의 예측 성과·희소성은 따로 판단하세요.

In [6]:
def evaluate(scored, name):
    """최상위 조합 부족과 검증 KPI를 별도로 집계한다."""
    n = len(scored)
    # '상위 정확히 20%'를 행 수로 정수화할 때 내림: 6,216*0.2=1,243.2 → 1,243건.
    k = int(np.floor(n*.20))
    # 위험도가 같으면 notice_key 순으로 순서를 고정해 재실행 결과를 일치시킵니다.
    top = scored.sort_values(['risk_score','notice_key'], ascending=[False,True]).head(k)
    return dict(model=name, features=' × '.join(LABELS[c] for c in BASE +
                                                ([] if name == '기준' else selected[:int(name.split('_')[-1])])),
                validation_n=n, exact_level_count=int(scored.basis_step.eq(0).sum()),
                fallback_count=int(scored.basis_step.gt(0).sum()),
                fallback_rate=float(scored.basis_step.gt(0).mean()),
                sparse_fallback_count=int(scored.fallback_reason.eq('최상위 조합 10건 미만').fillna(False).sum()),
                sparse_fallback_rate=float(scored.fallback_reason.eq('최상위 조합 10건 미만').fillna(False).mean()),
                invalid_fallback_count=int(scored.fallback_reason.eq('입력 오류/결측').sum()),
                agency_only_or_global_rate=float(scored.risk_basis_level.isin([LABELS['orntCode'],'전체']).mean()),
                median_basis_count=float(scored.risk_basis_count.median()),
                top20_n=k, top20_capture=float(top.is_failed.sum()/scored.is_failed.sum()) if scored.is_failed.sum() else np.nan,
                top20_failure_rate=float(top.is_failed.mean()),
                validation_failure_rate=float(scored.is_failed.mean()))

scored_models = {}
rows = []
for n_extra in range(len(selected)+1):
    name = '기준' if n_extra == 0 else f'추가_{n_extra}'
    scored_models[name] = score_by_history(train, valid, selected[:n_extra])
    rows.append(evaluate(scored_models[name], name))
comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))

model                                                    features  validation_n  exact_level_count  fallback_count  fallback_rate  sparse_fallback_count  sparse_fallback_rate  invalid_fallback_count  agency_only_or_global_rate  median_basis_count  top20_n  top20_capture  top20_failure_rate  validation_failure_rate
   기준                             기관코드 (기관그룹 대체) × 사업유형 × 계약/경쟁방식          6219               5573             646       0.103875                    646              0.103875                       0                    0.030069               545.0     1243       0.318624            0.342719                 0.214986
 추가_1                   기관코드 (기관그룹 대체) × 사업유형 × 계약/경쟁방식 × 공고기간 구간          6219               5313             906       0.145683                    901              0.144879                       5                    0.030069               484.0     1243       0.324607            0.349155                 0.214986
 추가_2         기관코드 (기관그룹 대체) × 사업유형 × 계약/경쟁방식 × 공고기간

## 5. 결과 파일 저장과 읽는 순서

아래 셀은 모든 결과를 `risk_review/`에 저장합니다.

| 파일 | 내용 |
|---|---|
| `chi_square_train.csv` | 변수별 검정 통계량, p값, Holm 보정 p값, 기대빈도 |
| `bin_failure_rates_train.csv` | 학습 자료의 구간별 공고 수와 유찰률 |
| `fallback_comparison_2025.csv` | 변수를 추가하기 전후의 대체율·사전 식별률 |
| `notice_risk_validation_2025.csv` | **공고별** 위험도, 적용 조합, 근거 건수, 대체 사유 |
| `risk_review.md` | PDF 문구 수정에 활용할 수치와 해석 |

공고별 CSV의 `is_failed`는 **검증 정답**이며 신규 공고 위험도 입력에 넣으면 미래 정보가 새어 들어갑니다.
추가 저장: `analysis_train_2021_2024.csv`, `analysis_valid_2025.csv`, `analysis_exclusion_summary.csv`, `excluded_notices.csv`도 같은 폴더에 저장합니다.


In [7]:
tests.to_csv(OUT/'chi_square_train.csv', index=False, encoding='utf-8-sig')
bin_rates = pd.concat([
    train.groupby(c, dropna=False)['is_failed'].agg(['count','sum','mean'])
         .reset_index().rename(columns={c:'category','count':'notice_count',
                                        'sum':'failed_count','mean':'failure_rate'})
         .assign(variable=LABELS[c])
    for c in ADDITIONAL], ignore_index=True)
bin_rates.to_csv(OUT/'bin_failure_rates_train.csv', index=False, encoding='utf-8-sig')
comparison.to_csv(OUT/'fallback_comparison_2025.csv', index=False, encoding='utf-8-sig')
predictions = valid[['notice_key','pblancDate','orntCode','busiDivs','cntrctMth',
                     'duration_bin','amount_bin','quarter_bin','is_failed']].copy()
for name, scored in scored_models.items():
    for col in ['risk_score','risk_basis_count','risk_basis_level','basis_step',
                'full_level_count','fallback_reason']:
        predictions[name+'_'+col] = scored[col].to_numpy()
predictions.to_csv(OUT/'notice_risk_validation_2025.csv', index=False, encoding='utf-8-sig')

def percent(v): return f'{v:.2%}'
base, final = comparison.iloc[0], comparison.iloc[-1]
lines = [
    '# 공고 시점 변수와 유찰 위험도 재검토', '',
    '2021~2024년 결과확정 경쟁입찰로 변수 검정과 유찰률표를 구축하고 2025년 공고로 평가.',
    f'기관그룹 매핑이 제공되지 않아 `orntCode`({train.orntCode.nunique():,}개 학습 기관코드)를 대체 기준으로 사용함. 문서의 기관그룹 기준 결과와 직접 비교할 수 없음.',
    '공고기간: 5일 이하 / 5일 초과~10일 / 10일 초과; 기초금액: 1천만원 이하 / 1천만원 초과~1억원 / 1억원 초과; 공고 시기: 분기.',
    '음수·결측 기간과 결측·오류 금액은 해당 변수의 검정에서 제외하고 위험도 산정에서는 상위 조합으로 대체.',
    '추가 후보 3개에 대해 카이제곱 독립성 검정의 p값을 Holm 보정하여 0.05 미만이면 채택. 연관성은 인과관계를 뜻하지 않음.',
    '', '## 검정 결과', '',
    '| 변수 | 학습 건수 | p값 | Holm 보정 p값 | Cramér V | 채택 |',
    '|---|---:|---:|---:|---:|---|',
]
for _, r in tests.iterrows():
    fmt = lambda v: '-' if pd.isna(v) else f'{v:.3g}'
    lines.append(f'| {r.variable} | {r.n:,} | {fmt(r.p_value)} | {fmt(r.holm_p_additional)} | {fmt(r.cramers_v)} | {"예" if r.selected_additional else "기존/아니요"} |')
lines += ['', '## 2025년 검증과 희소 조합', '',
          '| 모형 | 최상위 조합 10건 미만 대체 | 입력 오류/결측 대체 | 전체 대체 | 기관 단독/전체 적용 | 근거 건수 중앙값 | 상위 20% 유찰 사전 식별률 | 고위험군 유찰률 |',
          '|---|---:|---:|---:|---:|---:|---:|---:|']
for _,r in comparison.iterrows():
    lines.append(f'| {r.model} | {percent(r.sparse_fallback_rate)} ({r.sparse_fallback_count:,}/{r.validation_n:,}) | {r.invalid_fallback_count:,}건 | {percent(r.fallback_rate)} | {percent(r.agency_only_or_global_rate)} | {r.median_basis_count:,.0f} | {percent(r.top20_capture)} | {percent(r.top20_failure_rate)} |')
lines += ['', f'기준 대비 전체 추가 후 10건 미만 대체율 변화: {(final.sparse_fallback_rate-base.sparse_fallback_rate)*100:+.2f}%p.',
          f'2025년 전체 유찰률: {percent(final.validation_failure_rate)}; 상위 20%는 {int(final.top20_n):,}건(내림).',
          f'추가 선정 변수: {selected}. 기준 대비 최종 사전 식별률 변화: {(final.top20_capture-base.top20_capture)*100:+.2f}%p. 검증 결과는 재실행된 데이터에 따라 달라짐.',
          '최종 조합은 검토 대상이며 운영 채택은 표본 부족과 검증 성과를 함께 확인하여 결정할 것. 2025년 결과로 최종 조합을 선택한다면 별도의 후속 평가 기간이 필요함.',
          f'기대빈도 5 미만 셀이 있는 변수: {tests.loc[tests.expected_lt5.gt(0), "column"].tolist()}. 해당 변수의 근사 p값 해석에 주의가 필요함.',
          '구간 경계는 고정하고 추가 변수는 2021~2024년으로 선정함. 기준 3변수는 기존 방식과 비교하기 위해 유지하며, 비유의 변수가 있다면 운영 조합에서는 재검토해야 함.',
          '카이제곱의 유의성만으로 개선을 주장하지 않고 대체율과 홀드아웃 사전 식별률을 함께 확인할 것.', '']
lines += ['', '## 자료 준비 및 해석 범위', '', f'학습 {len(train):,}건, 검증 {len(valid):,}건. 낙찰·순위확정을 비유찰로 처리하며 순위확정은 최종 계약 체결을 뜻하지 않음.', '동일 noticeKey에 결과 상태가 여러 개인 공고는 판단보류. 계약정보·조달계획의 진행상태는 위험도 입력에 쓰지 않음.', '공고일 기준 연도 분리, 학습에는 2025년 이전 개찰 건만 사용. 결과 공개일은 제공되지 않아 공개 지연에 따른 시점 누수를 완전히 확인할 수 없음.', '조달계획·계약정보·수집로그는 이 위험도 계산에 필수 입력이 아님. 결과 미연결 공고는 비유찰로 간주하지 않고 제외함.', '같은 사업의 재공고가 서로 다른 기간에 포함될 수 있어 사업 단위 독립성이 보장되지는 않음.', '관측 가능한 결과가 있는 경쟁입찰만 분석하므로 모든 공고를 대표하는 유찰률이라고 단정할 수 없음.', '']
(OUT/'risk_review.md').write_text('\n'.join(lines), encoding='utf-8')
print('저장:', OUT.resolve())
print(f'전체 실행 시간: {perf_counter()-START:.2f}초')


저장: C:\Users\user\Downloads\risk_review
전체 실행 시간: 8.38초


## 결과를 해석할 때

세 추가 후보는 학습 자료의 카이제곱 검정에서 유의했습니다. 그러나 **유의함과 위험도 개선은 다릅니다.** 2025년 자료에서는 공고기간과 기초금액까지 추가할 때 사전 식별률이 높았고, 분기까지 추가하면 10건 미만 대체율이 더 커지며 식별률은 직전 조합보다 낮았습니다. 기관코드와 일부 계약방식에는 기대빈도 5 미만인 셀이 있어 해당 p값 해석에 주의해야 합니다.

이 노트북의 기관코드는 PDF의 기관그룹을 대신합니다. 기관그룹 매핑표가 확보되면 같은 절차를 다시 실행하여 원래 문서의 운영 모형을 평가해야 합니다.

| 변수 | 분석 건수 | p-value | Cramér's V | 판단 |
|---|---:|---:|---:|---|
| 기관코드 | 27,872 | <0.001 | 0.2327 | 기존 변수 |
| 사업유형 | 27,872 | <0.001 | 0.0422 | 기존 변수 |
| 계약·경쟁방식 | 27,872 | <0.001 | 0.0418 | 기존 변수 |
| 공고기간 | 27,868 | 0.002567 | 0.0207 | 추가 검토 |
| 기초금액 | 27,296 | <0.001 | 0.0572 | 추가 검토 |
| 공고 시기 | 27,872 | 0.000746 | 0.0246 | 추가 검토 |